In [1]:
!pip install torch-geometric
!pip install geoopt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 7.9 MB/s eta 0:00:00


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
import warnings
import geoopt  # The Riemannian geometry library

# Suppress yfinance warnings for cleaner output
warnings.filterwarnings("ignore")

# Hardware acceleration setup
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

In [3]:
# --- 1. MODEL ARCHITECTURE (TOPOLOGICAL UPGRADE: H-STDH) ---
class HyperbolicHypergraph(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, c=1.0):
        super(HyperbolicHypergraph, self).__init__()
        self.manifold = geoopt.PoincareBall(c=c)
        self.lin_transform = torch.nn.Linear(in_channels, hidden_channels)
        self.lin_out = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, x, incidence_matrix):
        # 1. Standard feature transformation
        x = self.lin_transform(x)
        x = F.leaky_relu(x, 0.2)

        # 2. Project to Poincaré Ball (Capture hierarchical hierarchy)
        x_hyp = self.manifold.expmap0(x)

        # 3. Map to Tangent Space for stable set-aggregation
        x_tangent = self.manifold.logmap0(x_hyp)

        # 4. Node-to-Hyperedge Aggregation (Create the cluster centers of mass)
        # Transpose incidence matrix to shape [Hyperedges, Nodes]
        H_t = incidence_matrix.t()
        # Normalize to prevent exploding gradients during high volatility
        H_t_norm = H_t / (H_t.sum(dim=1, keepdim=True) + 1e-8)
        hyperedge_features = torch.matmul(H_t_norm, x_tangent)

        # 5. Hyperedge-to-Node Aggregation (Pass cluster data back to individual stocks)
        H_norm = incidence_matrix / (incidence_matrix.sum(dim=1, keepdim=True) + 1e-8)
        node_features_updated = torch.matmul(H_norm, hyperedge_features)

        # 6. Final node-level prediction
        out = self.lin_out(node_features_updated)
        return out

In [4]:
def compute_hypergraph_incidence(df_train_returns, threshold=0.4):
    """
    Creates an Incidence Matrix [Nodes, Hyperedges].
    Each stock generates a hyperedge containing itself and all highly correlated peers.
    """
    corr_matrix = df_train_returns.corr().values
    num_nodes = corr_matrix.shape[0]

    # Initialize Incidence Matrix with zeros
    incidence_matrix = torch.zeros((num_nodes, num_nodes), dtype=torch.float)

    for i in range(num_nodes):
        for j in range(num_nodes):
            # If absolute correlation exceeds threshold, stock j belongs to hyperedge i
            if abs(corr_matrix[i, j]) > threshold:
                incidence_matrix[i, j] = 1.0

    # Fallback: Ensure no stock is left completely isolated (prevents matrix singularity)
    for i in range(num_nodes):
        if incidence_matrix[i].sum() == 0:
            incidence_matrix[i, i] = 1.0

    return incidence_matrix

In [5]:
def prepare_graph_snapshots(df_returns, lookback=5, scaler=None, is_train=True):
    if is_train:
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_returns.values)
    else:
        scaled_data = scaler.transform(df_returns.values)

    X_list, y_list = [], []

    for i in range(len(scaled_data) - lookback - 1):
        # Transpose to shape: [num_nodes, lookback]
        X_window = scaled_data[i : i + lookback].T
        # Target shape: [num_nodes, 1]
        y_window = scaled_data[i + lookback + 1].reshape(-1, 1)

        X_list.append(torch.tensor(X_window, dtype=torch.float))
        y_list.append(torch.tensor(y_window, dtype=torch.float))

    return X_list, y_list, scaler

In [6]:
# --- 4. THE PURGED WALK-FORWARD ENGINE ---
def run_walk_forward_backtest(df_returns, train_window=252, test_window=21, overlap_days=5, lookback=5):
    total_days = len(df_returns)
    start_idx = 0
    fold_count = 1

    print(f"Starting Purged Walk-Forward Engine on {device} (Total Trading Days: {total_days})\n")
    print("-" * 60)

    while start_idx + train_window + overlap_days + test_window <= total_days:
        train_start = start_idx
        train_end = start_idx + train_window
        df_train = df_returns.iloc[train_start:train_end]

        test_start = train_end + overlap_days
        test_end = test_start + test_window
        df_test = df_returns.iloc[test_start:test_end]

                # 1. Compute Hypergraph Topology & Send to GPU
        incidence_matrix = compute_hypergraph_incidence(df_train, threshold=0.4).to(device)

        # 2. Prepare Tensors
        X_train, y_train, scaler = prepare_graph_snapshots(df_train, lookback, is_train=True)
        X_test, y_test, _ = prepare_graph_snapshots(df_test, lookback, scaler=scaler, is_train=False)

        # 3. Initialize Model and Optimizer (TOPOLOGICAL UPGRADE)
        model = HyperbolicHypergraph(in_channels=lookback, hidden_channels=16, out_channels=1).to(device)

        optimizer = geoopt.optim.RiemannianAdam(model.parameters(), lr=0.005)
        criterion = torch.nn.MSELoss()

        # 4. Train the Model
        model.train()
        epochs = 30
        for epoch in range(epochs):
            epoch_loss = 0
            for i in range(len(X_train)):
                optimizer.zero_grad()

                x_in = X_train[i].to(device)
                y_target = y_train[i].to(device)

                # Pass the incidence matrix instead of the edge index
                out = model(x_in, incidence_matrix)
                loss = criterion(out, y_target)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()

        avg_train_loss = epoch_loss / len(X_train)

        # 5. Evaluate Out-of-Sample (OOS) & Calculate Sharpe
        model.eval()

        is_preds, is_targets = [], []
        oos_preds, oos_targets = [], []

        with torch.no_grad():
            for i in range(len(X_train)):
                x_in = X_train[i].to(device)
                y_target = y_train[i].to(device)
                out = model(x_in, incidence_matrix)
                is_preds.append(out)
                is_targets.append(y_target)

            for i in range(len(X_test)):
                x_in = X_test[i].to(device)
                y_target = y_test[i].to(device)
                out = model(x_in, incidence_matrix)
                oos_preds.append(out)
                oos_targets.append(y_target)
        # 6. Calculate Sharpe Ratios
        is_sharpe = calculate_sharpe_ratio(is_preds, is_targets, top_k=2)
        oos_sharpe = calculate_sharpe_ratio(oos_preds, oos_targets, top_k=2) if len(X_test) > 0 else 0

        print(f"Fold {fold_count:02d} | Train: {df_train.index[0].date()} to {df_train.index[-1].date()} | Test: {df_test.index[0].date()} to {df_test.index[-1].date()}")
        print(f"        | IS Sharpe:  {is_sharpe:.3f}")
        print(f"        | OOS Sharpe: {oos_sharpe:.3f}")
        print("-" * 60)

        start_idx += test_window
        fold_count += 1

In [7]:
def calculate_sharpe_ratio(predictions, targets, top_k=2):
    daily_portfolio_returns = []

    for i in range(len(predictions)):
        # Safe-guard: Detach from GPU (if applicable) before converting to NumPy
        pred = predictions[i].cpu().detach().numpy().flatten()
        actual = targets[i].cpu().detach().numpy().flatten()

        sort_idx = np.argsort(pred)
        short_idx = sort_idx[:top_k]
        long_idx = sort_idx[-top_k:]

        daily_ret = np.mean(actual[long_idx]) - np.mean(actual[short_idx])
        daily_portfolio_returns.append(daily_ret)

    daily_portfolio_returns = np.array(daily_portfolio_returns)
    mean_ret = np.mean(daily_portfolio_returns)
    std_ret = np.std(daily_portfolio_returns)

    if std_ret == 0:
        return 0.0

    # FIX 2: Hourly annualization factor (252 days * ~7 trading hours = 1764)
    sharpe = (mean_ret / std_ret) * np.sqrt(1764)
    return sharpe

In [ ]:
tickers = ['NVDA', 'AMD', 'INTC', 'TSM', 'AAPL', 'MSFT', 'GOOGL', 'AMZN']
print(f"Downloading hourly OHLCV data for: {tickers}...")

# FIX 1: Explicitly select ['Close'] to prevent multi-index matrix explosions
data = yf.download(tickers, period="730d", interval="1h", progress=False, auto_adjust=True)['Close']

df_returns = np.log(data / data.shift(1)).dropna()

# Adjusted for HOURLY data:
# Train window = 840 hours (~4 months)
# Test window = 140 hours (~3 weeks)
run_walk_forward_backtest(df_returns, train_window=840, test_window=140, overlap_days=5, lookback=5)

Starting Purged Walk-Forward Engine on cuda (Total Trading Days: 5070)

------------------------------------------------------------
Fold 01 | Train: 2023-08-23 to 2024-02-14 | Test: 2024-02-15 to 2024-03-15
        | IS Sharpe:  0.864
        | OOS Sharpe: 3.948
------------------------------------------------------------
Fold 02 | Train: 2023-09-21 to 2024-03-14 | Test: 2024-03-15 to 2024-04-15
        | IS Sharpe:  0.781
        | OOS Sharpe: -2.675
------------------------------------------------------------
Fold 03 | Train: 2023-10-19 to 2024-04-12 | Test: 2024-04-15 to 2024-05-13
        | IS Sharpe:  0.717
        | OOS Sharpe: 4.062
------------------------------------------------------------
Fold 04 | Train: 2023-11-16 to 2024-05-10 | Test: 2024-05-13 to 2024-06-11
        | IS Sharpe:  -1.262
        | OOS Sharpe: 1.488
------------------------------------------------------------
